# Sacks 2026 — All of Us run (HANDOFF_Milad.md, Part B)

**Restartable.** After the app is stopped or recreated, run every cell from the top, in order, up to the "Status" cell. Each cell skips what is already done:
- the repo is cloned only if missing and is then updated;
- the bundle is unpacked only if `resources/` is missing;
- the run folder (`~/cnv_anthro_run`) is restored from the bucket backup if the local disk was wiped;
- the pipeline stages run in the background (`nohup`) and leave a `.done` marker. `launch()` skips finished stages, and every finished stage is backed up to the bucket.

**Keep the notebook itself in the bucket** (`workspace-bucket/cnv_growth_bmi/`) so it survives the app.

**Egress rule.** No participant count from 1 to 20 may leave the Workbench, and that includes pasting into chat. The QC cells below print counts 1–20 as `<=20`. Paste only their output, never raw tables.

## 1. Settings (the only cell to edit)

In [ ]:
import os, re, sys, json, glob, hashlib, subprocess, time, shutil
import pandas as pd

REPO_URL   = 'https://github.com/jsebat/cnv-dosage-architecture.git'   # Jon 2026-11-19: code moved here (same history)
BRANCH     = 'main'
PIN_COMMIT = '4b64136'         # Jon 2026-11-18: sex strata fix (numeric match). ce1ce67 _nosd fix (additivity rerun done). sv/prep/main(all,eur) c4e09df; gridjoint 05fa65b; burden/main_rest b97150f

HOME = os.path.expanduser('~')                 # /home/dataproc on the Spark app, /home/jupyter on a plain JupyterLab app
REPO = f'{HOME}/repos/cnv-growth-bmi'
ROOT = f'{REPO}/AllOfUs'
CODE = f'{ROOT}/cnv_anthro'
RUN  = os.path.expanduser('~/cnv_anthro_run')      # = results_dir/export_dir parent in the config (local disk, fast)

BUCKET_MNT = f'{HOME}/workspace/workspace-bucket/cnv_growth_bmi'          # bucket as mounted on the VM
BUCKET_GS  = os.environ['WORKSPACE_BUCKET'].rstrip('/') + '/cnv_growth_bmi'     # same folder, gs:// form
BACKUP_GS  = f'{BUCKET_GS}/run_backup'                                          # mirror of ~/cnv_anthro_run

BUNDLE     = 'cnv_resources_sacks2026_2026-11-16.tar.gz'
BUNDLE_SHA = '27c8b6dc5c66b4cf52c07bc3d195b36cf0658fc1395294f5409215142d9df44b'

# --- CDR v9 genomic data (mounted data collection; found 2026-11-17) ---
V9 = f'{HOME}/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read'
SV_VCF_PATTERN = f'{V9}/structural_variants/vcf/full/AoU_srWGS_SV.v9.chr{{c}}.vcf.gz'   # one GATK-SV VCF per chromosome
ANCESTRY_SRC   = f'{V9}/snpindel/aux/ancestry/ancestry_preds.tsv'
SV_PARALLEL    = 22            # chromosomes converted at once

CFG_TEMPLATE = 'configs/aou_sacks2026.json'          # tracked in git, never edited
CFG          = 'configs/aou_sacks2026_filled.json'   # untracked copy with the two paths filled
MIN_CELL     = 20              # set 39 if the dissemination article asks for >= 40, then rerun export
GRID_JOBS    = 24

GP = os.environ.get('GOOGLE_PROJECT')
def sh(cmd, check=True):
    print('$', cmd); r = subprocess.run(cmd, shell=True, executable='/bin/bash')
    if check and r.returncode: raise RuntimeError(f'exit {r.returncode}: {cmd}')
    return r.returncode
for d in ['data', 'results', 'logs', 'stages']:
    os.makedirs(f'{RUN}/{d}', exist_ok=True)
print('CDR      ', os.environ.get('WORKSPACE_CDR'))
print('bucket   ', BUCKET_GS, '| mounted:', os.path.isdir(os.path.dirname(BUCKET_MNT)))
print('project  ', GP)

## 2. Code: clone or update the repo

In [ ]:
if not os.path.isdir(f'{REPO}/.git'):
    os.makedirs(os.path.dirname(REPO), exist_ok=True)
    sh(f'git clone -b {BRANCH} {REPO_URL} {REPO}')
else:
    # local edits (e.g. notebooks opened inside the repo) are stashed, not lost: `git stash list` shows them
    sh(f'cd {REPO} && (git diff --quiet && git diff --cached --quiet || git stash push -m "auto-stash $(date +%F_%T)")')
    # repo moved 2026-11-19 (cnv-growth-bmi:cnv-gwas -> cnv-dosage-architecture:main, same commits): repoint an old clone
    sh(f'cd {REPO} && [ "$(git remote get-url origin)" = "{REPO_URL}" ] || git remote set-url origin {REPO_URL}')
    sh(f'cd {REPO} && git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH}')
if PIN_COMMIT:
    sh(f'cd {REPO} && git checkout -q {PIN_COMMIT}')
COMMIT = subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--format=%h %cs %s']).decode().strip()
print('code at:', COMMIT)
open(f'{RUN}/logs/code_commits.txt', 'a').write(f'{time.strftime("%F %T")}\t{COMMIT}\n')
sh(f'cd {REPO} && git merge-base --is-ancestor dec364d HEAD && echo "OK: at or after dec364d"')

## 3. VM check
Needs 64–96 vCPUs, ≥128 GB RAM, ≥200 GB free disk. **Set the app's auto-pause / idle timeout to the maximum**: background jobs do not count as activity, and a paused VM kills them.

In [ ]:
sh('nproc; free -g | head -2; df -h $HOME | tail -1; python3 --version')
sh('python3 -c "import numpy, scipy, pandas, matplotlib; print(\'numpy\', numpy.__version__, \'scipy\', scipy.__version__, \'pandas\', pandas.__version__)"')
# BigQuery client (phenotype extraction in the prep steps). On the JupyterLab app (2026-11-20) the preinstalled
# google-cloud-bigquery was newer than google-api-core ("google.api_core has no attribute check_python_version"): upgrade both.
bq = subprocess.run(f'{sys.executable} -c "from google.cloud import bigquery; print(\'bigquery\', bigquery.__version__)"', shell=True, capture_output=True, text=True)
if bq.returncode != 0:
    print('BigQuery import failed:', bq.stderr.strip().splitlines()[-1] if bq.stderr.strip() else '?', '-> upgrading')
    sh(f'{sys.executable} -m pip install -q -U google-api-core google-cloud-bigquery db-dtypes')
    bq = subprocess.run(f'{sys.executable} -c "from google.cloud import bigquery; print(\'bigquery\', bigquery.__version__)"', shell=True, capture_output=True, text=True)
print(bq.stdout.strip() or bq.stderr.strip()[-500:])

## 4. Resources bundle (public annotations + UKBB shift lists)

In [ ]:
if not os.path.exists(f'{ROOT}/resources/shared_shifts/SHIFTS_MANIFEST.tsv'):
    local = f'{ROOT}/{BUNDLE}'
    if os.path.exists(f'{BUCKET_MNT}/{BUNDLE}'):
        shutil.copy(f'{BUCKET_MNT}/{BUNDLE}', local)
    else:
        sh(f'gsutil cp {BUCKET_GS}/{BUNDLE} {local}')
    h = hashlib.sha256(open(local, 'rb').read()).hexdigest()
    assert h == BUNDLE_SHA, f'sha256 mismatch: {h}'
    print('sha256 OK')
    sh(f'cd {ROOT} && tar xzf {BUNDLE} && rm -f {BUNDLE}')
print(sorted(os.listdir(f'{ROOT}/resources')))
man = pd.read_csv(f'{ROOT}/resources/shared_shifts/SHIFTS_MANIFEST.tsv', sep='\t', dtype=str)
print(man.to_string(index=False))
assert not man.apply(lambda c: c.str.strip().eq('NO')).any().any(), 'a shift list says NO: stop and tell Jon'

## 4b. Restore the run folder from the bucket backup (no-op on a fresh run)

In [ ]:
def backup(run=RUN, gs=BACKUP_GS):
    sh(f'gsutil -m -q rsync -r {run} {gs}')
# every dataset's run folder + the array inputs (same paths as DATASETS in section 7)
RESTORE = [(RUN, BACKUP_GS),
           (os.path.expanduser('~/cnv_anthro_nosegdup'), f'{BUCKET_GS}/run_backup_nosegdup'),
           (os.path.expanduser('~/cnv_anthro_array'), f'{BUCKET_GS}/run_backup_array'),
           (os.path.expanduser('~/cnv_anthro_array/inputs'), f'{BUCKET_GS}/array_inputs')]
for run_, gs_ in RESTORE:
    have_backup = subprocess.run(f'gsutil -q ls {gs_}/', shell=True, capture_output=True).returncode == 0
    # restore ONLY when the local copy is empty/new (no finished stage, no inputs): on a live VM the older backup would overwrite newer local logs
    has_local = bool(glob.glob(f'{run_}/stages/*.done')) or os.path.exists(f'{run_}/BUILD_INFO.txt')
    if have_backup and not has_local:
        os.makedirs(run_, exist_ok=True); sh(f'gsutil -m -q rsync -r {gs_} {run_}')   # adds/updates local files; deletes nothing
        print('restored', run_)
    elif have_backup:
        print('local copy has content, restore skipped:', run_)
    print('  stages done:', sorted(os.path.basename(p)[:-5] for p in glob.glob(f'{run_}/stages/*.done')) or 'none')

## 5. Data paths: check that the v9 SV VCFs (chr1–22) and ancestry_preds.tsv are visible

In [ ]:
miss = [c for c in range(1, 23) if not os.path.exists(SV_VCF_PATTERN.format(c=c))]
print('SV VCFs chr1-22:', 'all present' if not miss else f'MISSING {miss}')
print('ancestry_preds.tsv:', os.path.exists(ANCESTRY_SRC))
sh(f'ls -la {os.path.dirname(SV_VCF_PATTERN)} | head -5; du -ch {os.path.dirname(SV_VCF_PATTERN)}/*chr[0-9]*.vcf.gz | tail -1', check=False)
assert not miss and os.path.exists(ANCESTRY_SRC)

## 6. Fill the config (writes an untracked copy; the tracked template is not touched)
AoU v9 ships the SV VCF per chromosome, so the SV → CNV conversion runs as its own stage (`sv`: 22 chromosomes in parallel, same script and filters as the config, then concatenated). The PGC 10% rule is computed within chromosome and over the same sample count, so this equals a single-file run. The config's `prep` then only extracts the phenotypes.

In [ ]:
anc = f'{RUN}/data/ancestry_preds.tsv'
if not os.path.exists(anc):
    if ANCESTRY_SRC.startswith('gs://'):
        sh(f'gsutil -u {GP} cp {ANCESTRY_SRC} {anc}')
    else:
        shutil.copy(ANCESTRY_SRC, anc)
import sys
cfg = json.load(open(f'{CODE}/{CFG_TEMPLATE}'))
p0, p1 = cfg['prep_commands']
assert p0[0] == 'aou_sv_to_cnv.py' and p1[0] == 'aou_pheno_extract.py'
p1[p1.index('--ancestry') + 1] = anc
cfg['prep_commands'] = [p1]                       # SV conversion = stage 'sv' (per chromosome)
SV_OUT = p0[p0.index('--out') + 1].replace('${HOME}', os.path.expanduser('~'))
SV_SAMPLES = p0[p0.index('--write_samples') + 1].replace('${HOME}', os.path.expanduser('~'))
SV_ARGS = ' '.join(f'{p0[i]} {p0[i + 1]}' for i in range(1, len(p0), 2)
                   if p0[i] not in ('--vcf', '--out', '--write_samples', '--chroms'))
SVD = f'{RUN}/data/sv_by_chr'
sv_sh = f'''#!/bin/bash
# AoU v9 SV VCF per chromosome -> one CNV table (aou_sv_to_cnv.py, filters from the config: {SV_ARGS})
set -euo pipefail
mkdir -p {SVD}; cd {CODE}
one() {{
  c=$1
  [ -s {SVD}/chr$c.cnv ] && {{ echo "chr$c done"; return 0; }}
  {sys.executable} aou_sv_to_cnv.py --vcf {SV_VCF_PATTERN.format(c='$c')} --out {SVD}/chr$c.cnv.tmp \\
     --write_samples {SVD}/samples_chr$c.txt --chroms $c-$c {SV_ARGS} > {SVD}/chr$c.log 2>&1
  mv {SVD}/chr$c.cnv.tmp {SVD}/chr$c.cnv; echo "chr$c ok: $(tail -1 {SVD}/chr$c.log)"
}}
export -f one
seq 1 22 | xargs -P {SV_PARALLEL} -I{{}} bash -c 'one {{}}'
# every chromosome must list the same samples in the same order
for c in $(seq 2 22); do cmp -s {SVD}/samples_chr1.txt {SVD}/samples_chr$c.txt || {{ echo "sample list differs: chr$c"; exit 1; }}; done
cp {SVD}/samples_chr1.txt {SV_SAMPLES}
head -1 {SVD}/chr1.cnv > {SV_OUT}.tmp
for c in $(seq 1 22); do tail -n +2 {SVD}/chr$c.cnv >> {SV_OUT}.tmp; done
mv {SV_OUT}.tmp {SV_OUT}
echo "samples $(wc -l < {SV_SAMPLES}); calls $(( $(wc -l < {SV_OUT}) - 1 ))"
'''
open(f'{RUN}/logs/sv_stage.sh', 'w').write(sv_sh)
cfg['min_cell'] = MIN_CELL
assert 'FILL_FROM_DATA_DICTIONARY' not in json.dumps(cfg['prep_commands'])
json.dump(cfg, open(f'{CODE}/{CFG}', 'w'), indent=1)
shutil.copy(f'{CODE}/{CFG}', f'{RUN}/logs/config_used.json')
print('sv stage ', f'{RUN}/logs/sv_stage.sh', '| filters:', SV_ARGS)
print('cnv table', SV_OUT); print('ancestry ', anc, f'({sum(1 for _ in open(anc)) - 1} rows)')
print('min_cell ', cfg['min_cell']); print('config   ', f'{CODE}/{CFG}')

## 7. Run the pipeline in the background

| stage | steps | time |
|---|---|---|
| `sv` | 22 SV VCFs → CNV table (parallel; finished chromosomes are kept on restart) | ~1–3 h |
| `prep` | OMOP → phenotypes (BigQuery) | < 1 h |
| `main` | gwas, plots, alleles, loci, burden (all 9 strata) | hours |
| `additivity` | additivity, scores (stratum all) | minutes |
| `gridjoint` | grid, joint (stratum all), 1,000–2,000 nulls | several hours |
| `export` | small-cell suppression → `export_aggregate/` (run **after QC**, section 9) | minutes |

`launch()` runs the listed stages in order, skips `.done` ones, stops at the first failure, and backs up `~/cnv_anthro_run` to the bucket after each stage. The jobs survive closing the browser or the kernel, but not a stopped VM.

In [ ]:
STAGES = {
    'sv':         None,                      # bash logs/sv_stage.sh (section 6)
    'prep':       '--steps prep',
    'main':       '--steps gwas,plots,alleles,loci,burden',   # FAILED at afr on c4e09df; all+eur gwas/alleles/loci valid. Superseded by the next two
    'burden':     '--steps burden',                                                   # b97150f: all 9 strata (WGS single_def changed)
    'main_rest':  '--steps gwas,plots,alleles,loci --strata afr,amr,eas,sas,mid,female,male',
    'sexstrata':  '--steps burden,gwas,plots,alleles,loci --strata female,male',   # 4b64136: female/male were N = 0 (sex 1.0/2.0)
    'additivity': '--steps additivity,scores --strata all',
    'gridjoint':  f'--steps grid,joint --strata all --jobs {GRID_JOBS}',
    'export':     '--steps export',
    'burden_allcalls': 'py:burden_allcalls.py',    # arrays (Jon 2026-11-20): second burden version, single_def all_calls -> <stratum>/burden_all_calls/<trait>
}
# Three datasets, each with its own run folder, filled config and bucket backup (stage markers live in <run>/stages):
#   wgs      = default WGS run (B2)          nosegdup = WGS without the segdup GWAS filter (B6)          array = Molly's array calls (B5)
DATASETS = {
    'wgs':      dict(run=RUN, cfg=CFG, backup=BACKUP_GS, tar='aou_sacks2026',
                     stages=['sv', 'prep', 'burden', 'main_rest', 'sexstrata', 'additivity', 'gridjoint', 'export']),   # 'main' (FAILED at afr, c4e09df) superseded by burden/main_rest/sexstrata
    'nosegdup': dict(run=os.path.expanduser('~/cnv_anthro_nosegdup'), cfg='configs/aou_sacks2026_nosegdup_filled.json',
                     backup=f'{BUCKET_GS}/run_backup_nosegdup', tar='aou_sacks2026_nosegdup', stages=['main', 'additivity', 'gridjoint', 'export']),
    'array':    dict(run=os.path.expanduser('~/cnv_anthro_array'), cfg='configs/aou_sacks2026_array_filled.json',
                     backup=f'{BUCKET_GS}/run_backup_array', tar='aou_array_sacks2026', stages=['prep', 'main', 'burden_allcalls', 'additivity', 'gridjoint', 'export']),
}
for D in DATASETS.values():
    for d in ['data', 'results', 'logs', 'stages']: os.makedirs(f"{D['run']}/{d}", exist_ok=True)
# drivers: tag '' = main, 'b' = second WGS driver, 'ns' = nosegdup, 'ar' = array. A driver runs one dataset; pid files stay in ~/cnv_anthro_run/logs
def pidf(tag=''): return f'{RUN}/logs/driver{tag}.pid'

def running(tag=''):
    if not os.path.exists(pidf(tag)): return None
    try:
        pid = int(open(pidf(tag)).read()); os.kill(pid, 0); return pid
    except (ValueError, ProcessLookupError, PermissionError):
        return None

def launch(stages, tag='', ds='wgs'):
    assert not running(tag), f'driver{tag} already running (pid {running(tag)}); see status()'
    PIDF = pidf(tag); D = DATASETS[ds]; RUN_, CFG_, BK_ = D['run'], D['cfg'], D['backup']
    assert os.path.exists(f'{CODE}/{CFG_}'), f'filled config missing: {CFG_} (run its fill cell first)'
    import sys
    lines = ['#!/bin/bash', f'echo $$ > {PIDF}', f'cd {CODE}', 'export PYTHONUNBUFFERED=1', f'export NUMEXPR_MAX_THREADS={os.cpu_count()}']
    for st in stages:
        lines += [f'if [ -e {RUN_}/stages/{st}.done ]; then echo "skip {ds}:{st} (done)"; else',
                  f'  rm -f {RUN_}/stages/{st}.failed; echo "=== {ds}:{st} start $(date)"; t0=$(date +%s)',
                  (f'  bash {RUN_}/logs/sv_stage.sh > {RUN_}/logs/{st}.log 2>&1' if STAGES[st] is None else
                   f'  {sys.executable} {RUN_}/logs/{STAGES[st][3:]} {CFG_} > {RUN_}/logs/{st}.log 2>&1' if STAGES[st].startswith('py:') else
                   f'  {sys.executable} run_pipeline.py --config {CFG_} {STAGES[st]} > {RUN_}/logs/{st}.log 2>&1'),
                  f'  rc=$?; echo "=== {ds}:{st} exit $rc after $(( ($(date +%s)-t0)/60 )) min $(date)"',
                  f'  if [ $rc -ne 0 ]; then touch {RUN_}/stages/{st}.failed; exit $rc; fi',
                  f'  echo "{COMMIT}" > {RUN_}/stages/{st}.done',
                  f'  gsutil -m -q rsync -r {RUN_} {BK_} >> {RUN_}/logs/backup.log 2>&1 && echo "backup ok"',
                  'fi']
    open(f'{RUN}/logs/driver{tag}.sh', 'w').write('\n'.join(lines) + '\n')
    if os.path.exists(PIDF): os.remove(PIDF)
    subprocess.Popen(f'nohup setsid bash {RUN}/logs/driver{tag}.sh >> {RUN}/logs/driver{tag}.log 2>&1 &', shell=True)
    time.sleep(2)
    print('started', ds, stages, 'driver', repr(tag), 'pid', running(tag), '\nwatch with status()')

def status(n=15):
    TAGS = ['', 'b', 'ns', 'ar']
    print('drivers:', ' | '.join(f"{t or 'main'}: {running(t) or '-'}" for t in TAGS))
    for ds, D in DATASETS.items():
        marks = {os.path.basename(p).rsplit('.', 1)[0]: p.rsplit('.', 1)[1] for p in glob.glob(f"{D['run']}/stages/*.done") + glob.glob(f"{D['run']}/stages/*.failed")}
        print(f'  {ds:9s}', '  '.join(f"{st}:{'DONE' if marks.get(st) == 'done' else 'FAILED' if marks.get(st) == 'failed' else '-'}" for st in D['stages']))
    sh(' '.join(f"tail -n 3 {RUN}/logs/driver{t}.log 2>/dev/null;" for t in TAGS), check=False)
    logs = sorted(sum((glob.glob(f"{D['run']}/logs/*.log") for D in DATASETS.values()), []), key=os.path.getmtime)
    logs = [l for l in logs if not os.path.basename(l).startswith(('driver', 'backup'))]
    # egress rule: raw logs contain model tables with small counts, so only progress/command/error lines are shown
    keep = re.compile(r'^\[ *[0-9.]+s\]|^=+ |^\+ |^== |Error|Traceback|FAILED|failed strata')
    for l in logs[-3:]:
        lines = [x.rstrip() for x in open(l, errors='replace') if keep.search(x)]
        print(f'\n--- {os.path.relpath(l, os.path.expanduser("~"))} (last {n} progress lines)'); print('\n'.join(x[:300] for x in lines[-n:]))
    sh('uptime; free -g | sed -n 2p', check=False)

In [ ]:
# WGS is finished (sv, prep, burden, main_rest, sexstrata, additivity, gridjoint). Do NOT relaunch: 'main' has no .done
# (it failed and was replaced by burden + main_rest), so launching it would rerun the whole WGS main step over the restored results.
# Original first launch, kept for the record:
# launch(['sv', 'prep', 'main', 'additivity', 'gridjoint'])

In [ ]:
status()      # rerun any time

In [ ]:
# manual backup any time (also runs automatically after each stage)
backup()

## 7b. Pinning a new commit mid-run (e.g. Jon's b97150f)
1. Set `PIN_COMMIT` in section 1, then rerun sections **1, 2, 4 and 6**. Section 6 must be rerun: the filled config is regenerated from the new template. A running stage is not affected when only scripts it does not call changed (b97150f changes `dup_burden.py` and `run_pipeline.py`, not the grid/joint scripts), but check `git diff --stat` first.
2. Run only what Jon asked for. `burden` and `main_rest` write `burden/` and the afr…male folders, never `all/grid` or `all/joint`, so they may run as a second driver while `gridjoint` runs (cell below).

In [ ]:
# ce1ce67 (_nosd reference fix): keep the pre-fix additivity results, then rerun additivity+scores (stratum all)
PRE = f'{RUN}/prefix_nosd_20261117'
if not os.path.exists(PRE):
    shutil.copytree(f'{RUN}/results/all/additivity', f'{PRE}/additivity'); shutil.copytree(f'{RUN}/results/all/scores', f'{PRE}/scores')
    print('pre-fix copy ->', PRE)
MARK = f'{RUN}/stages/additivity_rerun_ce1ce67.started'
if not os.path.exists(MARK):                     # one-time: done 2026-11-18, so this cell is a no-op on a rerun
    if os.path.exists(f'{RUN}/stages/additivity.done'): os.remove(f'{RUN}/stages/additivity.done')
    open(MARK, 'w').write('ce1ce67\n'); launch(['additivity'], tag='b')
else:
    print('ce1ce67 additivity rerun already done/started:', os.path.exists(f'{RUN}/stages/additivity.done') and open(f'{RUN}/stages/additivity.done').read().strip())

In [ ]:
# sex coding check: female/male strata came out N = 0 (sex written as 1.0/2.0?)
ph = pd.read_csv(f'{RUN}/data/aou_pheno_std.tsv', sep='\t', usecols=['sex'], dtype=str)
print('sex values as written:', mask(ph.sex.value_counts().to_dict()))

In [ ]:
# b97150f: burden in all strata, then gwas/plots/alleles/loci in afr..male (second driver, alongside gridjoint)
launch(['burden', 'main_rest'], tag='b')

In [ ]:
# 4b64136 (Jon 2026-11-18): make_stratum matches sex numerically, so the existing aou_pheno_std.tsv works (no prep rerun).
# Reruns burden + gwas/plots/alleles/loci for female and male only; other strata untouched. Expect N = 57,965 / 37,058 (before trait filters).
assert COMMIT.startswith('4b64136') or PIN_COMMIT == '4b64136', 'pin 4b64136 first (sections 1, 2, 4, 6)'
launch(['sexstrata'], tag='b')

In [ ]:
# check after sexstrata finishes: stratum N and the outputs exist (aggregate only)
R = f'{RUN}/results'
for st in ['female', 'male']:
    sh(f"grep -h '== stratum {st}' {RUN}/logs/sexstrata.log | tail -1", check=False)
    for t in ['height', 'bmi']:
        fs = [f'{R}/{st}/taskA_gwas/{t}', f'{R}/{st}/taskA_alleles/{t}/allele_assoc.tsv', f'{R}/{st}/burden/{t}/run_info.json', f'{R}/{st}/locus_effects/locus_effects_{t}.tsv']
        print(f'  {st} {t}:', ['ok' if os.path.exists(f) else 'MISSING ' + os.path.relpath(f, R) for f in fs])

## 8. QC (B3) — outputs are small-cell masked and safe to paste
Sections 8–10 run on one dataset at a time: set `DS` in the next cell (`'wgs'`, `'nosegdup'` or `'array'`) and rerun 8–10. 8.6 is WGS only.

In [ ]:
def mask(o):
    # counts 1..20 -> '<=20' (egress rule); applied to every int in the printed output
    if isinstance(o, dict): return {k: mask(v) for k, v in o.items()}
    if isinstance(o, list): return [mask(v) for v in o]
    if isinstance(o, bool): return o
    if isinstance(o, (int,)) and 1 <= o <= 20: return '<=20'
    if isinstance(o, float) and o.is_integer() and 1 <= o <= 20: return '<=20'
    return o
def flat(d, pre=''):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict): out.update(flat(v, f'{pre}{k}.'))
        elif not isinstance(v, list): out[f'{pre}{k}'] = v
    return out
DS = 'wgs'                       # 'wgs' | 'nosegdup' | 'array'
DRUN = DATASETS[DS]['run']; R = f'{DRUN}/results'; print('QC/export dataset:', DS, '->', R)
def show_info(pattern, keys=r'(^|\.)(N|n_|n$|carrier|union|asym|hit_)', maxk=25):
    import re
    for f in sorted(glob.glob(f'{R}/{pattern}', recursive=True)):
        d = flat(json.load(open(f)))
        d = {k: v for k, v in d.items() if re.search(keys, k)}
        print(os.path.relpath(f, R), json.dumps(mask(dict(list(d.items())[:maxk]))))

# 8.0 phenotype summary from prep (already suppressed by the extractor)
p = f'{R}/aou_pheno_summary.json'
if os.path.exists(p): print(json.dumps(mask(json.load(open(p))), indent=1))

In [ ]:
# 8.1 sample sizes: GWAS and burden per stratum, and the carrier subsets
show_info('*/taskA_gwas/*/run_info.json')
show_info('*/burden/*/run_info.json')
show_info('*/joint/strata/union_strata_info.json')

In [ ]:
# 8.2 the 'only' additivity runs -> send Jon reference_N and locus_carriers_in_sample (main: no_rare_gene; sensitivity: _no_rare_gene_nosd)
rows = []
for f in sorted(glob.glob(f'{R}/all/additivity/*/only*/run_info.json')):
    d = flat(json.load(open(f))); parts = os.path.relpath(f, R).split('/')
    g = lambda k: next((v for kk, v in d.items() if kk == k or kk.endswith('.' + k)), None)   # keys are nested in run_info
    rows.append(dict(trait=parts[2], run=parts[3], N=g('N'), reference_N=g('reference_N'),
                     locus_carriers_in_sample=g('locus_carriers_in_sample'), only_ref=g('only_ref')))
T_only = pd.DataFrame(rows)
for c_ in ['N', 'reference_N', 'locus_carriers_in_sample']:
    if c_ in T_only: T_only[c_] = T_only[c_].map(lambda v: mask(v) if isinstance(v, (int, float)) and v == v else v)
print(T_only.to_string(index=False))

In [ ]:
# 8.2b signed-burden panel (b97150f): send Jon single_dosage_N for all/height (with reference_N from 8.2)
for st in ['all']:
    for t in ['height', 'bmi']:
        f = f'{R}/{st}/burden/{t}/run_info.json'
        if not os.path.exists(f): print('missing', f); continue
        d = flat(json.load(open(f)))
        print(st, t, json.dumps(mask({k: v for k, v in d.items() if re.search(r'single|min_n|^N$', k)})))

In [ ]:
# 8.3 hit carriers in the additivity runs: hit_carriers_DEL / _DUP must be > 20
import re
for f in sorted(glob.glob(f'{R}/all/additivity/**/run_info.json', recursive=True)):
    d = flat(json.load(open(f)))
    hc = {k: v for k, v in d.items() if re.search(r'hit_carriers', k)}
    bad = [k for k, v in hc.items() if isinstance(v, (int, float)) and v <= 20]
    print('FLAG ' if bad else 'ok   ', os.path.relpath(f, R), json.dumps(mask(hc)))

In [ ]:
# 8.4 recurrent-CNV carrier frequencies (EUR and all): paste this output, it is compared with UKBB outside the Workbench
REC = r'15q11\.2_BP1\.BP2_(DEL|DUP)$|16p11\.2_BP4\.BP5_(DEL|DUP)$|16p11\.2_BP1\.BP4|22q11\.21_A\.D_(DEL|DUP)$|1q21\.1|17q12'
for st in ['eur', 'all']:
    f = f'{R}/{st}/taskA_alleles/height/allele_assoc.tsv'
    if not os.path.exists(f): print('missing', f); continue
    d = pd.read_csv(f, sep='\t').drop_duplicates('allele')
    d = d[d.allele.str.contains(REC)][['allele', 'n_carriers', 'N', 'FREQ']].copy()
    small = d.n_carriers.between(1, 20)
    d['n_carriers'] = d.n_carriers.astype(object); d.loc[small, 'n_carriers'] = '<=20'; d.loc[small, 'FREQ'] = None
    print(f'--- {st}'); print(d.to_string(index=False))

In [ ]:
# 8.5 Miami plots (probe-level, aggregate): genomic inflation per stratum
from IPython.display import Image, display
for f in sorted(glob.glob(f'{R}/*/taskA_gwas/*/miami_*.png')):
    print(os.path.relpath(f, R)); display(Image(f, width=900))

In [ ]:
# 8.6 1q21.1 TAR DUP breakpoint check (Jon 2026-11-18). All DUP calls in chr1:145.3-146.3 Mb, people in the phenotype table.
# Aggregate only: sites with <= 20 carriers are pooled, counts 1-20 masked, no margins, totals rounded to 10 (no masked cell
# can be recovered by subtraction). Safe to paste.
r10 = lambda n: f'~{int(round(n, -1))}' if n > 20 else mask(int(n))
import numpy as np
SD = f'{CODE}/../resources/segDups.bed'
TAR = {'BP1-BP2': (145627235, 145809100), 'BP2-BP3': (145823900, 146040039), 'BP1-BP3': (145627235, 146040039)}
ids = set(pd.read_csv(f'{RUN}/data/aou_pheno_std.tsv', sep='\t', usecols=['IID'], dtype=str).IID)
cnv = pd.read_csv(f'{RUN}/data/aou_srwgs_sv.cnv', sep='\t', dtype={'IID': str})
c = cnv[(cnv.CHR == 1) & (cnv.TYPE == 3) & (cnv.BP2 > 145_300_000) & (cnv.BP1 < 146_300_000) & cnv.IID.isin(ids)].copy()
print('DUP calls in window:', r10(len(c)), '| carriers:', r10(c.IID.nunique()), '| distinct sites (BP1,BP2):', c.groupby(['BP1', 'BP2']).ngroups)

sd = pd.read_csv(SD, sep='\t', header=None, usecols=[0, 1, 2], names=['chr', 's', 'e'])
sd = sd[sd.chr.astype(str).isin(['chr1', '1'])]
mid = ((c.BP1 + c.BP2) // 2).to_numpy()
c['mid_in_segdup'] = [bool(((sd.s <= m) & (sd.e >= m)).any()) for m in mid]
seg_cov = lambda a, b: (lambda x: x[x[:, 0] < x[:, 1]])(np.c_[np.maximum(sd.s.to_numpy(), a), np.minimum(sd.e.to_numpy(), b)])
def frac_sd(a, b):            # share of the call covered by the union of segdups
    iv = seg_cov(a, b)
    if not len(iv): return 0.0
    iv = iv[np.argsort(iv[:, 0])]; tot, cs, ce = 0, iv[0, 0], iv[0, 1]
    for x, y in iv[1:]:
        if x > ce: tot += ce - cs; cs, ce = x, y
        else: ce = max(ce, y)
    return (tot + ce - cs) / (b - a)
ov = lambda a, b, s, e: max(0, min(b, e) - max(a, s))
def labels(a, b):
    L = b - a
    recip = [k for k, (s, e) in TAR.items() if ov(a, b, s, e) >= 0.5 * L and ov(a, b, s, e) >= 0.5 * (e - s)]
    cover = [k for k, (s, e) in TAR.items() if k != 'BP1-BP3' and ov(a, b, s, e) >= 0.5 * (e - s)]   # alleles.py rule (coverF)
    pipe = 'BP1-BP3' if set(cover) == {'BP1-BP2', 'BP2-BP3'} else (cover[0] if cover else '-')
    return ','.join(recip) or 'none', pipe

# site types from the chr1 SV VCF via its .tbi index (pysam; the VM has no tabix binary). Records are keyed by POS = BP1.
vcf = SV_VCF_PATTERN.format(c=1); st = {}
try:
    try:
        import pysam
    except ImportError:
        subprocess.run(f'{sys.executable} -m pip install -q pysam', shell=True, check=True); import pysam
    tb = pysam.TabixFile(vcf); ctg = 'chr1' if 'chr1' in tb.contigs else '1'
    lo, hi = int(c.BP1.min()) - 1, int(c.BP1.max()) + 1
    for ln in tb.fetch(ctg, lo, hi):
        f = ln.split('\t', 8); info = dict(x.split('=', 1) for x in f[7].split(';') if '=' in x)
        st[(int(f[1]), int(info.get('END', f[1])))] = f"{info.get('SVTYPE')}/{f[6]}"
    print('VCF records read:', len(st))
except Exception as e:
    print('VCF lookup skipped:', repr(e))

S = c.groupby(['BP1', 'BP2']).agg(carriers=('IID', 'nunique'), mid_in_segdup=('mid_in_segdup', 'first')).reset_index()
S['kb'] = (S.BP2 - S.BP1) // 1000
S[['recip50', 'pipeline_label']] = [labels(a, b) for a, b in zip(S.BP1, S.BP2)]
S['segdup_frac'] = [round(frac_sd(a, b), 2) for a, b in zip(S.BP1, S.BP2)]
S['vcf_type'] = [st.get((a, b), '?') for a, b in zip(S.BP1, S.BP2)]

print('\n--- by pipeline label (alleles.py coverF rule, before person-level collapsing) x 50% reciprocal match')
g = c.merge(S[['BP1', 'BP2', 'recip50', 'pipeline_label', 'vcf_type']], on=['BP1', 'BP2'])
print(mask(pd.crosstab([g.pipeline_label, g.recip50], g.vcf_type).to_dict('index')))
print('\n--- midpoint in a segdup, by pipeline label (calls)'); print(mask(pd.crosstab(g.pipeline_label, g.mid_in_segdup).to_dict('index')))

print('\n--- start / end distribution (25 kb bins, Mb; calls)')
for col in ['BP1', 'BP2']:
    b = (c[col] // 25000 * 25000 / 1e6).round(3).value_counts().sort_index()
    print(col, {float(k): int(v) for k, v in b.items() if v > 20}, '| other bins pooled:', r10(int(b[b <= 20].sum())))

print('\n--- sites with >= 21 carriers (others pooled below)')
big = S[S.carriers > 20].sort_values('BP1')
print(big[['BP1', 'BP2', 'kb', 'carriers', 'recip50', 'pipeline_label', 'mid_in_segdup', 'segdup_frac', 'vcf_type']].to_string(index=False))
sm = S[S.carriers <= 20]
print(f'pooled small sites: {len(sm)} sites, carriers', r10(int(sm.carriers.sum())), '| by pipeline label:',
      {k: r10(int(v)) for k, v in sm.groupby('pipeline_label').carriers.sum().items()})

In [ ]:
# 8.7 ARRAY ONLY: are recurrent DUPs/DELs split or truncated? (2026-11-20: array DUP carrier frequencies at 22q11.21 A-D,
# 1q21.1, TAR BP1-BP3 and 17q12 are well below UKBB, and TAR BP1-BP2 DUP is above). alleles.py needs ONE call covering >= 50%
# of the locus; consStart/consEnd is where both callers agree, so a call can be shorter than either caller's.
# Per locus and type, people in the phenotype table with any call overlapping >= 10% of the locus, by:
#   best1 = coverage by their single best call, union = coverage by all their calls in the locus merged; ncalls = calls in locus.
# Aggregate only: counts <= 20 masked, others rounded to 10, no totals. Safe to paste.
assert DS == 'array', 'set DS = "array" in section 8 first'
r10 = lambda n: f'~{int(round(n, -1))}' if n > 20 else mask(int(n))
LOCI = {'22q11.21_A-D': (22, 19049819, 21112437), '1q21.1': (1, 147056425, 147922330), 'TAR_BP1-BP3': (1, 145627235, 146040039),
        'TAR_BP1-BP2': (1, 145627235, 145809100), '17q12': (17, 36460090, 37857812), '15q11.2_BP1-BP2': (15, 22778537, 23067754),
        '16p11.2_BP4-BP5 (control)': (16, 29639519, 30189452)}
ids = set(pd.read_csv(f'{DRUN}/data/aou_array_pheno_std.tsv', sep='\t', usecols=['IID'], dtype=str).IID)
cnv = pd.read_csv(f'{DRUN}/data/aou_array.cnv', sep=r'\s+', dtype={'IID': str})
cnv = cnv[cnv.IID.isin(ids)]
def union_len(iv):
    iv = sorted(iv); tot, cs, ce = 0, iv[0][0], iv[0][1]
    for x, y in iv[1:]:
        if x > ce: tot += ce - cs; cs, ce = x, y
        else: ce = max(ce, y)
    return tot + ce - cs
rows = []
for name, (ch, a, b) in LOCI.items():
    L = b - a
    for typ, tname in [(1, 'DEL'), (3, 'DUP')]:
        c = cnv[(cnv.CHR == ch) & (cnv.TYPE == typ) & (cnv.BP2 > a) & (cnv.BP1 < b)].copy()
        if not len(c): rows.append(dict(locus=name, type=tname, people_ge10pct='0')); continue
        c['s'], c['e'] = c.BP1.clip(lower=a), c.BP2.clip(upper=b)
        g = pd.DataFrame([{'best1': (d.e - d.s).max() / L, 'union': union_len(list(zip(d.s, d.e))) / L, 'ncalls': len(d)} for _, d in c.groupby('IID')])
        g = g[g.union >= 0.10]
        if not len(g): rows.append(dict(locus=name, type=tname, people_ge10pct='0')); continue
        rows.append(dict(locus=name, type=tname, people_ge10pct=r10(len(g)),
                         best1_ge50=r10((g.best1 >= .5).sum()), best1_20_50=r10(((g.best1 >= .2) & (g.best1 < .5)).sum()),
                         best1_lt20=r10((g.best1 < .2).sum()), union_ge50=r10((g.union >= .5).sum()),
                         split_2plus_calls=r10((g.ncalls >= 2).sum()), median_best1_cov=round(float(g.best1.median()), 2) if len(g) > 20 else 'masked'))
print(pd.DataFrame(rows).fillna('').to_string(index=False))
# raw PGC call columns (header only): are there per-caller or union coordinates besides consStart/consEnd?
RAW = os.path.expanduser('~/cnv_anthro_array/molly_raw')          # section 12's raw copy (restored from the array backup)
plt_ = sorted(glob.glob(f'{RAW}/4_filtered/*.plt'))[:1]
print('\nraw .plt header:', open(plt_[0]).readline().strip().split('\t') if plt_ else f'no .plt under {RAW}/4_filtered')

## 9. Export (after QC; dataset = `DS` from section 8). Check the in-Workbench article *Downloading and Disseminating Genomic Data*: if it asks for ≥ 40, set `MIN_CELL = 39` in section 1, rerun sections 1 and 6, delete `stages/export.done`, and tell Jon.

In [ ]:
launch(['export'], tag={'wgs': '', 'nosegdup': 'ns', 'array': 'ar'}[DS], ds=DS)

In [ ]:
status()
E = f'{DRUN}/export_aggregate'
if os.path.exists(f'{E}/EXPORT_MANIFEST.txt'): print(open(f'{E}/EXPORT_MANIFEST.txt').read()[:3000])

In [ ]:
# 9.1 audit: no count from 1 to 20 in any count-like column of any exported table
import re
COUNT = re.compile(r'^(n|N)$|^n_|_n$|^N_|carrier|count|^ncnv|^n[A-Z]', re.I)
# not person counts (2026-11-20 audit): probes per call, z statistics of the no-loci runs, mean/SD of per-person feature counts
NOT_COUNT = re.compile(r'^n_probes$|^noloci_|^(mean|sd)_count$', re.I)
bad, nfile = [], 0
for f in glob.glob(f'{E}/**/*.tsv', recursive=True) + glob.glob(f'{E}/**/*.csv', recursive=True):
    nfile += 1
    d = pd.read_csv(f, sep='\t' if f.endswith('.tsv') else ',', dtype=str, low_memory=False)
    for c in [c for c in d.columns if COUNT.search(str(c)) and not NOT_COUNT.search(str(c))]:
        v = pd.to_numeric(d[c], errors='coerce')
        k = int(v.between(1, 20).sum())
        if k: bad.append((os.path.relpath(f, E), c, k))
print(nfile, 'tables checked')
print('\n'.join(f'VIOLATION {f} [{c}] {k} rows' for f, c, k in bad) or 'OK: no 1-20 counts in count columns')
sh(f'find {E} -type f | grep -i -E "pheno|genotype|\\.cnv" || echo "OK: no person-level file names"', check=False)
sh(f'du -sh {E}; find {E} -name null_runs.npz | wc -l')

## 10. Package for UCSD (B4; dataset = `DS`)
Ask support@researchallofus.org for the current egress size limit **before** downloading. Then download the two files from the Jupyter file browser (the dataset's run folder, e.g. `~/cnv_anthro_run/`) and `scp` them to the shared results folder at UCSD.

In [ ]:
T = f'{DATASETS[DS]["tar"]}_export_{time.strftime("%Y%m%d")}.tar.gz'
assert not bad, 'fix the export audit first'
sh(f'cd {DRUN} && tar czf {T} export_aggregate && sha256sum {T} > {T}.sha256 && ls -la {T}* && cat {T}.sha256')
sh(f'gsutil cp {DRUN}/{T} {DRUN}/{T}.sha256 {BUCKET_GS}/')

## 11. Array inputs from Molly's workspace ("PGC CNV project") — look only, nothing is copied yet
Her workspace shows a migrated Researcher-Workbench bucket (`rw-migration-aou-rw-<id>`) and the **C2022Q4R13 (v7)** CDR. Find the array CNV calls, the QC'd sample list and the batch file, then fill the three paths in section 12 (next notebook version).

In [ ]:
sh('wb workspace list 2>&1 | head -40', check=False)        # note the ID of "PGC CNV project"

In [ ]:
MOLLY_WS = 'FILL: workspace id of "PGC CNV project" from the list above'
sh(f'wb resource list --workspace={MOLLY_WS} 2>&1', check=False)
sh(f'wb resource describe --workspace={MOLLY_WS} --name=<migrated bucket resource name> 2>&1', check=False)   # gives the gs:// bucket name

In [ ]:
MOLLY_GS = 'gs://FILL-bucket-name-from-describe'
sh(f'gsutil -u {os.environ["GOOGLE_PROJECT"]} ls {MOLLY_GS}/ 2>&1 | head -50', check=False)
# then drill down; file names and sizes only (no head/cat of participant-level files into chat):
# sh(f'gsutil -u {os.environ["GOOGLE_PROJECT"]} ls -lr {MOLLY_GS}/<folder>/ | grep -i -E "cnv|penncnv|quantisnp|sample|batch|qc" | head -50')

## 12. Build the three array inputs from Molly's PGC CNV calls (B5)
Source: `results/4_filtered/GDA-8v1-0/*.plt` (PGC CNV pipeline, consensus calls, 448 batches), sample QC `Freq_Calc == 1` in `results/QC/GDA-8v1-0/GDA-8v1-0.txt`, batch from `results/sample_info/GDA-8v1-0.txt`. Breakpoints = `consStart/consEnd`; only consensus calls (`algorithm` = IP,UP: PennCNV (UP) + QuantiSNP (IP in this project) agree, per Jeff); `4_filtered` = locus merge + sample QC only, so all size/probe/frequency filtering happens in `filter_plink_cnv.py`; TYPE 1 = DEL, 3 = DUP (DELDUP = caller disagreement, dropped); SITES = probes from `probes.txt` inside the call. Everything is written to the VM and mirrored to our own bucket (`array_inputs/`), nothing is printed per person.

In [ ]:
MOLLY_GS = os.environ.get('MOLLY_GS', 'gs://FILL-bucket-of-the-PGC-CNV-workspace')   # set MOLLY_GS (bucket of Molly's PGC CNV workspace)
P = os.environ['GOOGLE_PROJECT']
ARUN = os.path.expanduser('~/cnv_anthro_array'); AIN = f'{ARUN}/inputs'; RAW = f'{ARUN}/molly_raw'
for d in [AIN, f'{RAW}/4_filtered', f'{ARUN}/data', f'{ARUN}/results']: os.makedirs(d, exist_ok=True)
if not glob.glob(f'{RAW}/4_filtered/*.plt'):
    sh(f'gsutil -u {P} -m -q cp {MOLLY_GS}/results/4_filtered/GDA-8v1-0/*.plt {RAW}/4_filtered/')
for f in ['results/QC/GDA-8v1-0/GDA-8v1-0.txt', 'results/sample_info/GDA-8v1-0.txt', 'inputs/probes.txt']:
    out = f'{RAW}/' + f.replace('/', '__')
    if not os.path.exists(out): sh(f'gsutil -u {P} -q cp {MOLLY_GS}/{f} {out}')
print(len(glob.glob(f'{RAW}/4_filtered/*.plt')), 'call files'); sh(f'du -sh {RAW}')

In [ ]:
import numpy as np
if 'mask' not in globals():   # defined in section 8; needed here when section 8 was not run in this kernel
    def mask(o):
        if isinstance(o, dict): return {k: mask(v) for k, v in o.items()}
        if isinstance(o, bool): return o
        if isinstance(o, (int, float)) and float(o).is_integer() and 1 <= o <= 20: return '<=20'
        return o
norm = lambda s: pd.to_numeric(s.astype(str).str.replace('chr', '', regex=False).replace({'X': '23', 'Y': '24'}), errors='coerce')

# samples passing QC + batch
qc = pd.read_csv(f'{RAW}/results__QC__GDA-8v1-0__GDA-8v1-0.txt', sep='\t', usecols=['identifier', 'Freq_Calc'], dtype=str)
si = pd.read_csv(f'{RAW}/results__sample_info__GDA-8v1-0.txt', sep='\t', usecols=['#batch', 'identifier'], dtype=str)
passing = qc.loc[qc.Freq_Calc == '1', 'identifier']
passing.to_csv(f'{AIN}/array_samples_qc_pass.txt', index=False, header=False)
si[['identifier', '#batch']].to_csv(f'{AIN}/array_batch.tsv', sep='\t', index=False, header=False)

# calls
c = pd.concat([pd.read_csv(f, sep='\t', usecols=['chrom', 'consStart', 'consEnd', 'consCallType', 'algorithm', 'sampleId'], dtype={'sampleId': str})
               for f in sorted(glob.glob(f'{RAW}/4_filtered/*.plt'))], ignore_index=True)
n_raw = len(c)
# algorithm x call type (aggregate; masked)
print(mask(pd.crosstab(c.algorithm, c.consCallType).to_dict()))  # no margins: a total minus the shown cells would reveal a masked one
# keep IP,UP consensus only (both callers), whatever the separator/order
both = c.algorithm.astype(str).str.upper().str.split(r'[,;+ ]+').apply(lambda a: {'IP', 'UP'} <= set(a))
n_nonconsensus = int((~both).sum()); c = c[both]
c['CHR'] = norm(c.chrom); c['TYPE'] = c.consCallType.map({'DEL': 1, 'DUP': 3})
c = c.dropna(subset=['CHR', 'TYPE']).astype({'CHR': int, 'TYPE': int, 'consStart': int, 'consEnd': int})

# SITES = probes inside [BP1, BP2]
pr = pd.read_csv(f'{RAW}/inputs__probes.txt', sep='\t', dtype={'Chromosome': str})
pr['CHR'] = norm(pr.Chromosome); pr = pr.dropna(subset=['CHR'])
c['SITES'] = 0
for ch, g in pr.groupby('CHR'):
    pos = np.sort(g.Position.astype(int).values); m = c.CHR == int(ch)
    c.loc[m, 'SITES'] = np.searchsorted(pos, c.loc[m, 'consEnd'].values, 'right') - np.searchsorted(pos, c.loc[m, 'consStart'].values, 'left')

out = pd.DataFrame({'FID': c.sampleId, 'IID': c.sampleId, 'CHR': c.CHR, 'BP1': c.consStart, 'BP2': c.consEnd,
                    'TYPE': c.TYPE, 'SCORE': 0, 'SITES': c.SITES})
out.to_csv(f'{AIN}/aou_array_pgc.cnv', sep='\t', index=False)

# aggregate checks only
print('calls read', n_raw, '| non-IP,UP dropped', mask(n_nonconsensus), '| DEL/DUP kept', len(out),
      '| DELDUP or bad chrom dropped', mask(n_raw - n_nonconsensus - len(out)))
print('samples: QC-pass', len(passing), '| with batch', si.identifier.nunique(), '| batches', si['#batch'].nunique())
print('call IDs not in sample_info:', mask(int((~out.IID.isin(si.identifier)).sum())),
      '| calls from QC-failed samples:', mask(int((~out.IID.isin(passing)).sum())))
print('SITES quantiles (5/50/95%):', out.SITES.quantile([.05, .5, .95]).tolist(), '| share of calls with 0 probes:', round((out.SITES == 0).mean(), 4))
print('share >= 10 kb:', round(((out.BP2 - out.BP1) >= 10000).mean(), 3), '| share >= 10 kb and >= 10 probes:',
      round((((out.BP2 - out.BP1) >= 10000) & (out.SITES >= 10)).mean(), 3))
open(f'{AIN}/BUILD_INFO.txt', 'w').write(f'{time.strftime("%F %T")}\tIP,UP consensus only; DEL/DUP; consStart/consEnd; SITES from probes.txt\n')
sh(f'ls -la {AIN}; gsutil -m -q rsync -r {AIN} {BUCKET_GS}/array_inputs && echo "mirrored to {BUCKET_GS}/array_inputs"')

In [ ]:
# preview of Jon's B5 calls-per-person line (same script and filters as the array prep step; temp output only)
sh(f'cd {CODE} && {sys.executable} filter_plink_cnv.py --cnv {AIN}/aou_array_pgc.cnv --samples {AIN}/array_samples_qc_pass.txt '
   f'--exclude_samples {RUN}/data/sv_samples.txt --out {ARUN}/data/preview.cnv --write_samples {ARUN}/data/preview_samples.txt '
   f'--min_size 10000 --min_probes 10 --max_freq 0.10 2>&1 | tail -3')

In [ ]:
# 12b. calls-per-person profile under stricter filters (aggregate only: medians and percentages; temp output deleted).
# GDA-8v1-0 has ~1.9M probes vs ~0.82M on the UKBB Axiom array, so 10 probes on GDA span far less DNA than 10 probes on UKBB;
# ~25 GDA probes is roughly the UKBB 10-probe resolution. UKBB reference: median 1 call/person, ~21% with none.
for size, probes, freq in [(10000, 10, 0.10), (10000, 25, 0.10), (20000, 20, 0.10), (50000, 25, 0.10), (10000, 10, 0.01), (20000, 25, 0.01)]:
    print(f'--- >= {size // 1000} kb, >= {probes} probes, freq <= {freq:.0%}', flush=True)
    sh(f'cd {CODE} && {sys.executable} filter_plink_cnv.py --cnv {AIN}/aou_array_pgc.cnv --samples {AIN}/array_samples_qc_pass.txt '
       f'--exclude_samples {RUN}/data/sv_samples.txt --out {ARUN}/data/diag.cnv --write_samples {ARUN}/data/diag_samples.txt '
       f'--min_size {size} --min_probes {probes} --max_freq {freq} 2>&1 | tail -1 | grep -o "calls/person.*"')
for f in ['diag.cnv', 'diag_samples.txt']:
    if os.path.exists(f'{ARUN}/data/{f}'): os.remove(f'{ARUN}/data/{f}')

## 13. WGS without the segdup filter (B6)
Same CNV and phenotype tables as the WGS run (no prep of its own); only `gwas.segdups = null`. Writes to `~/cnv_anthro_nosegdup/`, backup `run_backup_nosegdup/`. Runs on driver `'ns'`, so it can run alongside the array run (watch memory in `status()`; lower `GRID_JOBS` in section 1 if both grids run at once).

In [ ]:
cfg = json.load(open(f'{CODE}/configs/aou_sacks2026_nosegdup.json'))
assert cfg['prep_commands'] == [] and cfg['gwas']['segdups'] is None
cfg['min_cell'] = MIN_CELL
json.dump(cfg, open(f"{CODE}/{DATASETS['nosegdup']['cfg']}", 'w'), indent=1)
shutil.copy(f"{CODE}/{DATASETS['nosegdup']['cfg']}", f"{DATASETS['nosegdup']['run']}/logs/config_used.json")
assert os.path.exists(os.path.expanduser(cfg['cnv_table'].replace('${HOME}', '~'))) and os.path.exists(os.path.expanduser(cfg['pheno_table'].replace('${HOME}', '~'))), 'WGS sv/prep tables missing'
print('nosegdup config ->', DATASETS['nosegdup']['cfg'], '| results', cfg['results_dir'], '| min_cell', cfg['min_cell'])

In [ ]:
# all strata: gwas, plots, alleles, loci, burden; then additivity/scores and grid/joint (stratum all). Export after QC (section 8 with DS = 'nosegdup').
launch(['main', 'additivity', 'gridjoint'], tag='ns', ds='nosegdup')

## 14. AoU array dataset (B5)
Inputs from section 12 (rerun it first: IP,UP consensus calls only). The prep step runs `filter_plink_cnv.py` (drops WGS participants; >= 10 kb, >= 10 probes, PGC 10% rule) and the phenotype extractor (BigQuery) with genotyping batch as a covariate. Writes to `~/cnv_anthro_array/`, backup `run_backup_array/`, driver `'ar'`. Jon 2026-11-20 (cell 12b: almost everyone has a call on this array): the 'only' runs use `no_rare_gene` as the primary reference and `nocall` as the sensitivity (`_nocall`); the burden panel uses `rare_gene` (`burden/`) plus an `all_calls` version (`burden_all_calls/`, its own stage).

In [ ]:
AIN = os.path.expanduser('~/cnv_anthro_array/inputs')
assert os.path.exists(f'{AIN}/BUILD_INFO.txt') and 'IP,UP' in open(f'{AIN}/BUILD_INFO.txt').read(), 'rerun section 12 (consensus filter) first'
cfg = json.load(open(f'{CODE}/configs/aou_sacks2026_array.json'))
f0, f1 = cfg['prep_commands']
assert f0[0] == 'filter_plink_cnv.py' and f1[0] == 'aou_pheno_extract.py'
f0[f0.index('--cnv') + 1] = f'{AIN}/aou_array_pgc.cnv'
f0[f0.index('--samples') + 1] = f'{AIN}/array_samples_qc_pass.txt'
f1[f1.index('--ancestry') + 1] = f'{RUN}/data/ancestry_preds.tsv'
f1[f1.index('--batch') + 1] = f'{AIN}/array_batch.tsv'
cfg['min_cell'] = MIN_CELL
# Jon 2026-11-20: keep the B5 call filters; median 8 calls/person and 0.1% with no call (UKBB 1 and ~21%), so
# - 'only' runs: primary reference = no rare gene-hitting call (WGS rule); 'nocall' = sensitivity (runs suffixed _nocall)
# - burden panel: primary single_def = rare_gene (-> burden/); all_calls = second version (stage burden_allcalls -> burden_all_calls/)
cfg['figures']['only_ref'] = 'no_rare_gene'
cfg['figures']['only_ref_sensitivity'] = ['nocall']
cfg['figures'].setdefault('rare_freq', 0.01)
cfg['burden']['single_def'] = 'rare_gene'
cfg['burden'].setdefault('rare_freq', 0.01)
# second burden version: same dup_burden.py call as run_pipeline.py (same pheno, covariates, strata), only --single_def all_calls
open(f"{DATASETS['array']['run']}/logs/burden_allcalls.py", 'w').write('''import json, os, subprocess, sys
sys.path.insert(0, os.getcwd())
from run_pipeline import expand
C = expand(json.load(open(sys.argv[1]))); B = C['burden']; root = C['results_dir']; n = 0
for sname, filt in C['strata'].items():
    out = os.path.join(root, sname); pheno = os.path.join(out, 'stratum_pheno.tsv') if filt else C['pheno_table']
    for t, T in C['traits'].items():
        if not os.path.isdir(f'{out}/burden/{t}'):
            print('skip (no primary burden run):', sname, t, flush=True); continue
        fac = ','.join(f for f in T['factors'] if not (f == 'sex' and 'sex' in (filt or {})))
        cmd = [sys.executable, 'dup_burden.py', '--pheno', pheno, '--trait', T['column'], '--cnv', C['cnv_table'], '--exons', B['exons'],
               '--gene_cov', B['gene_cov'], '--clingen', B['clingen'], '--asym_loci', B['asym_loci'], '--numeric', ','.join(T['numeric']),
               '--factors', fac, '--out', f'{out}/burden_all_calls/{t}', '--min_n', str(B.get('min_n', 50)), '--single_def', 'all_calls',
               '--rare_freq', str(B.get('rare_freq', 0.01))]
        print('+', ' '.join(cmd), flush=True); subprocess.run(cmd, check=True); n += 1
print('burden_allcalls runs:', n)
''')
cfg['note'] += " Jon 2026-11-20: 'only'-run reference = no_rare_gene (nocall as sensitivity); burden single_def = rare_gene (+ all_calls in burden_all_calls/)."
assert 'FILL' not in json.dumps(cfg['prep_commands']), [x for x in json.dumps(cfg['prep_commands']).split('"') if 'FILL' in x]   # the note mentions FILL_, so check only the paths
for p_ in [f'{AIN}/aou_array_pgc.cnv', f'{AIN}/array_samples_qc_pass.txt', f'{AIN}/array_batch.tsv', f'{RUN}/data/ancestry_preds.tsv', f'{RUN}/data/sv_samples.txt']:
    assert os.path.exists(p_), p_
json.dump(cfg, open(f"{CODE}/{DATASETS['array']['cfg']}", 'w'), indent=1)
shutil.copy(f"{CODE}/{DATASETS['array']['cfg']}", f"{DATASETS['array']['run']}/logs/config_used.json")
print('array config ->', DATASETS['array']['cfg'], '| min_cell', cfg['min_cell'], '| only_ref', cfg['figures']['only_ref'],
      '+', cfg['figures']['only_ref_sensitivity'], '| burden', cfg['burden']['single_def'], '+ all_calls (burden_all_calls/)')
print('prep 1:', ' '.join(f0)); print('prep 2:', ' '.join(f1))

In [ ]:
# prep -> all strata -> additivity/scores -> grid/joint. Export after QC (section 8 with DS = 'array').
launch(['prep', 'main', 'burden_allcalls', 'additivity', 'gridjoint'], tag='ar', ds='array')

In [ ]:
# after the array prep stage: calls per person (send Jon this line) and the phenotype summary (both aggregate)
A = DATASETS['array']['run']
sh(f"grep -h -i -E 'calls|person|samples|none' {A}/logs/prep.log | grep -v -E '^\\s*$' | tail -8", check=False)
p_ = f'{A}/results/aou_pheno_summary.json'
if os.path.exists(p_): print(json.dumps(mask(json.load(open(p_))), indent=1) if 'mask' in globals() else 'run section 8 first for mask()')

## 15. Recurrent CNV genotyping with Molly's method (Jon 2026-11-23)
Port of `recurrentCNVgenotyping/GenotypeRecurrentCNVs_tutorial.ipynb` (same locus sheet, same 50% reciprocal / one-way rules, same segment patterns; no merging of fragmented calls). Runs on the WGS and array call tables and the people analysed. The genotype matrices stay in each run's `data/` folder (participant level, never exported). The cell prints masked DEL/DUP counts per locus next to the pipeline's `alleles.py` counts (rounded to 10). Safe to paste.

In [ ]:
# write the port and Molly's two locus tables (saved from her public sheet; the Workbench has no Google access)
MG = os.path.expanduser('~/recurrent_cnv_genotyping'); os.makedirs(MG, exist_ok=True)
for f, txt in {'recurrent_cnv_genotyping_aou.py': '#!/usr/bin/env python3\n"""Recurrent CNV genotyping for All of Us -- a port of Molly\'s recurrentCNVgenotyping/GenotypeRecurrentCNVs_tutorial.ipynb.\n\nSame locus definitions (her public sheet, saved as recurrentCNV_hg38.reciprocal.tsv / .oneway.tsv), same rules:\n  * reciprocal loci : a call is assigned if it overlaps the locus by >= 50% of the locus AND >= 50% of the call\n                      (bedtools intersect -f .5 -r)\n  * one-way loci    : a call is assigned if it covers >= 50% of the locus segment (bedtools intersect -f .5);\n                      segments are then combined into allele-specific genotypes by the exact patterns of the\n                      notebook (1q21.1 TAR, 15q11-13, 16p11.2, 16p12.2, 16p13.11, 22q11.21 A-D and D-H).\n  * genotype codes  : 1 = DEL, 2 = no call, 3 = DUP; when a person has several calls at one locus the first one wins\n                      (pivot_table aggfunc=\'first\', calls in input order), as in the notebook.\nFragmented calls are kept as they are (no merging), as in the notebook: e.g. a 22q11.21 pattern \'323\' gets no A-D genotype.\n\nDifferences from the notebook (mechanics only, no rule changes):\n  * no bedtools / Google Sheets: the overlap is computed in pandas with the same half-open arithmetic\n    (overlap = min(end) - max(start)), and the two locus tabs are read from local TSVs (the Workbench has no\n    Google access);\n  * input is the pipeline\'s PLINK-CNV table (FID IID CHR BP1 BP2 TYPE ...; TYPE 1 = DEL, 3 = DUP), so the same\n    script runs on AoU srWGS calls (aou_sv_to_cnv.py output) and AoU array calls (filter_plink_cnv.py output);\n  * the genotype matrix is restricted to --samples (people analysed); no sample-QC NA step (calls are already QC\'d).\nCross-check (crosscheck_molly.py: the notebook run cell by cell with its bedtools v2.28.0 binary, 4,000 simulated people,\n~10,700 carrier genotypes): identical except (a) people with both a DEL and a DUP at one locus, where "first call" depends\non bedtools\' output order, and (b) the notebook reads each bedtools output with pd.read_csv\'s default header=0, so the\nfirst hit of each intersect is lost (one person per run); the port keeps it.\n\nOutputs: --out genotype matrix (participant level: keep inside the Workbench) and, with --counts, the per-locus\nDEL/DUP carrier counts with 1..min_cell masked (aggregate; safe to export).\n"""\nimport argparse\nimport numpy as np\nimport pandas as pd\n\n# ---- combination patterns, copied from the notebook (segment order matters) ----\nPATTERNS = {\n    # 15q11-13 BP1-BP3: segments 15q11.2_BP1_BP2, 15q11.2-13.1_BP2_BP3\n    "15q13": (["15q11.2_BP1_BP2", "15q11.2-13.1_BP2_BP3"], {\n        "15q11.2_BP1_BP2": ("12", "32"), "15q11.2-13.1_BP2_BP3": ("21", "23"), "15q11.2-13.1_BP1_BP3": ("11", "33")}),\n    # 15q13 BP3-BP5: segments BP3_BP4, BP4_BP5_exclCHRNA7, CHRNA7\n    "15q35": (["15q13.1-13.2_BP3_BP4", "15q13.3_BP4_BP5_exclCHRNA7", "15q13.3_CHRNA7"], {\n        "15q13.1-13.2_BP3_BP4": ("122", "322"), "15q13.1-13.3_BP3_BP5": ("111", "333"), "15q13.3_BP4_BP5": ("211", "233"),\n        "15q13.3_BP4_BP5_exclCHRNA7": ("212", "232"), "15q13.3_CHRNA7": ("221", "223")}),\n    "16p11.2": (["16p11.2_BP1_BP2", "16p11.2_BP2_BP3", "16p11.2_BP3_BP4", "16p11.2_BP4_BP5"], {\n        "16p11.2_BP1_BP2": ("1222", "3222"), "16p11.2_BP1_BP3": ("1122", "3322"), "16p11.2_BP2_BP3": ("2122", "2322"),\n        "16p11.2_BP3_BP4": ("2212", "2232"), "16p11.2_BP4_BP5": ("2221", "2223")}),\n    "22q11.21_AD": (["22q11.21_A_B", "22q11.21_B_C", "22q11.21_C_D"], {\n        "22q11.21_A_B": ("122", "322"), "22q11.21_B_C": ("212", "232"), "22q11.21_C_D": ("221", "223"),\n        "22q11.21_A_C": ("112", "332"), "22q11.21_B_D": ("211", "233"), "22q11.21_A_D": ("111", "333")}),\n    "22q11.21_DH": (["22q11.21_D_E", "22q11.21_E_F", "22q11.21_F_G", "22q11.21_G_H"], {\n        "22q11.21_D_E": ("1222", "3222"), "22q11.21_E_F": ("2122", "2322"), "22q11.21_F_G": ("2212", "2232"),\n        "22q11.21_G_H": ("2221", "2223"), "22q11.21_F_H": ("2211", "2233"), "22q11.21_E_H": ("2111", "2333"),\n        "22q11.21_D_H": ("1111", "3333")}),\n}\n# three-breakpoint regions genotyped with np.where in the notebook (1q21.1 TAR, 16p12.2, 16p13.11)\nTRIPLES = [("1q21.1_TAR", "BP1_BP2", "BP2_BP3"), ("16p12.2", "BP1_BP2", "BP2_BP3"), ("16p13.11", "BP1_BP2", "BP2_BP3")]\n\n\ndef read_loci(path):\n    d = pd.read_csv(path, sep="\\t")\n    d = d.rename(columns={"Locus name": "NAME"})[["CHR", "START", "END", "NAME"]]\n    d["CHR"] = pd.to_numeric(d.CHR.astype(str).str.replace("chr", "", regex=False), errors="coerce")\n    return d.dropna(subset=["CHR"]).astype({"CHR": int, "START": int, "END": int})\n\n\ndef assign(calls, loci, reciprocal):\n    """genotype per (IID, locus) for the calls that pass the overlap rule; first call (input order) wins."""\n    out = []\n    for L in loci.itertuples():\n        c = calls[calls.CHR == L.CHR]\n        o = np.maximum(0, np.minimum(c.BP2.to_numpy(), L.END) - np.maximum(c.BP1.to_numpy(), L.START))\n        m = (o > 0) & (o >= 0.5 * (L.END - L.START))\n        if reciprocal:\n            m &= o >= 0.5 * (c.BP2 - c.BP1).to_numpy()\n        h = c[m]\n        out.append(pd.DataFrame({"IID": h.IID.to_numpy(), "locus": L.NAME, "CN": h.TYPE.to_numpy(), "ord": h.ord.to_numpy()}))\n    a = pd.concat(out, ignore_index=True).sort_values("ord", kind="mergesort")\n    return a.drop_duplicates(["IID", "locus"], keep="first")\n\n\ndef genotype(cnv, samples, rec, one):\n    cnv = cnv[cnv.TYPE.isin([1, 3]) & cnv.IID.isin(set(samples))].copy()\n    cnv["ord"] = np.arange(len(cnv))\n    G = pd.DataFrame(index=pd.Index(samples, name="IID"))\n    for loci, recip, suffix in [(rec, True, ""), (one, False, "_oneway")]:\n        a = assign(cnv, loci, recip)\n        for name in loci.NAME:\n            G[name + suffix] = 2\n        if len(a):\n            w = a.pivot_table(index="IID", columns="locus", values="CN", aggfunc="first")\n            for name in w.columns:\n                v = w[name].dropna()\n                G.loc[v.index, name + suffix] = v.astype(int).to_numpy()\n    # allele-specific genotypes from the one-way segments\n    for (reg, a1, a2) in TRIPLES:\n        s1, s2 = G[f"{reg}_{a1}_oneway"], G[f"{reg}_{a2}_oneway"]\n        G[f"{reg}_{a1}"] = np.where(s2 == 2, s1, 2)\n        G[f"{reg}_{a2}"] = np.where(s1 == 2, s2, 2)\n        G[f"{reg}_BP1_BP3"] = np.where(s1 == s2, s2, 2)\n    for key, (segs, alleles) in PATTERNS.items():\n        pat = G[[s + "_oneway" for s in segs]].astype(int).astype(str).agg("".join, axis=1)\n        for allele, (pdel, pdup) in alleles.items():\n            G[allele] = np.select([pat == pdel, pat == pdup], [1, 3], default=2)\n    return G\n\n\ndef counts(G, min_cell):\n    rows = []\n    for col in [c for c in G.columns if not c.endswith("_oneway")]:\n        nd, nu = int((G[col] == 1).sum()), int((G[col] == 3).sum())\n        m = lambda n: f"<={min_cell}" if 1 <= n <= min_cell else n\n        rows.append(dict(locus=col, n_DEL=m(nd), n_DUP=m(nu), N=len(G)))\n    return pd.DataFrame(rows)\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--cnv", required=True, help="PLINK-CNV table: FID IID CHR BP1 BP2 TYPE ... (TYPE 1 = DEL, 3 = DUP)")\n    ap.add_argument("--samples", required=True, help="one IID per line, or a TSV with an IID column (people analysed)")\n    ap.add_argument("--reciprocal", required=True, help="recurrentCNV_hg38.reciprocal.tsv (Molly\'s sheet, reciprocal tab)")\n    ap.add_argument("--oneway", required=True, help="recurrentCNV_hg38.oneway.tsv (Molly\'s sheet, one-way tab)")\n    ap.add_argument("--out", required=True, help="genotype matrix, IID x locus, codes 1/2/3 (participant level)")\n    ap.add_argument("--counts", default=None, help="per-locus DEL/DUP counts, masked (aggregate)")\n    ap.add_argument("--min_cell", type=int, default=20)\n    a = ap.parse_args()\n    s = pd.read_csv(a.samples, sep="\\t", dtype=str)\n    samples = list(s["IID"] if "IID" in s.columns else pd.read_csv(a.samples, header=None, dtype=str)[0])\n    cnv = pd.read_csv(a.cnv, sep=r"\\s+", dtype={"FID": str, "IID": str})\n    cnv["CHR"] = pd.to_numeric(cnv.CHR.astype(str).str.replace("chr", "", regex=False), errors="coerce")\n    G = genotype(cnv.dropna(subset=["CHR"]).astype({"CHR": int}), samples, read_loci(a.reciprocal), read_loci(a.oneway))\n    G.to_csv(a.out, sep="\\t")\n    if a.counts:\n        counts(G, a.min_cell).to_csv(a.counts, sep="\\t", index=False)\n    print(f"genotyped {len(G)} people at {sum(not c.endswith(\'_oneway\') for c in G.columns)} loci -> {a.out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'recurrentCNV_hg38.reciprocal.tsv': 'Group (in Figure 2A)\tCHR\tSTART\tEND\tLocus name\tDisorder Name\tLocus type\tURL for Clinical Reference\nGenome-wide significant loci\t1\t147056425\t147922330\t1q21.1\t1q21.1 recurrent region (distal)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37421\nGenome-wide significant loci\t3\t196011534\t197617305\t3q29\t3q29 recurrent region (includes DLG1)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37443\nGenome-wide significant loci\t7\t73331712\t74715504\t7q11.23_WBS\t7q11.23 recurrent (Williams-Beuren syndrome) region (includes ELN)\t\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37392\nOther Clinically-Reportable CNVs\t2\t96076661\t97011779\t2q11.2\t2q11.2 deletion (LMAN2L, ARID5A)\tRecurrent CNV\thttps://www.sciencedirect.com/science/article/pii/S2949774425011495#:~:text=2%20are%20approximately%20%E2%88%BC1.0,2%20rCNVs.\nOther Clinically-Reportable CNVs\t2\t110105139\t110226371\t2q13_NPHP1\t2q13 deletion (NPHP1)\tRecurrent CNV\thttps://rarechromo.org/media/information/Chromosome%20%202/2q13%20microdeletions%20FTNW.pdf\nOther Clinically-Reportable CNVs\t2\t110636463\t111255072\t2q13\t2q13 deletion (NPHP1)\tRecurrent CNV\thttps://rarechromo.org/media/information/Chromosome%20%202/2q13%20microdeletions%20FTNW.pdf\nOther Clinically-Reportable CNVs\t2\t130723735\t131173104\t2q21.1\t2q21.1 deletion\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-46288\nOther Clinically-Reportable CNVs\t7\t75508972\t76435095\t7q11.23_distal\tWBS distal deletion\tRecurrent CNV\thttps://pmc.ncbi.nlm.nih.gov/articles/PMC3865388/\nOther Clinically-Reportable CNVs\t10\t48182156\t49850750\t10q11.21_11.23\t10q11.21-11.23\tRecurrent CNV\thttps://pmc.ncbi.nlm.nih.gov/articles/PMC3655525/\nOther Clinically-Reportable CNVs\t10\t80285716\t87171894\t10q22-23\t10q23 deletion (NRG3, GRID1)\tseparate the genes?\thttps://pmc.ncbi.nlm.nih.gov/articles/PMC1852738/\nOther Clinically-Reportable CNVs\t13\t22981219\t24310484\t13q12.12\t13q12.12 deletion\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-46298\nOther Clinically-Reportable CNVs\t17\t14238070\t15523647\t17p12\tHNPP (PMP22)\tRecurrent CNV\t\nOther Clinically-Reportable CNVs\t17\t16909457\t20307704\t17p11.2\t17p11.2 recurrent (SMS/PLS) region (includes RAI1)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37418\nOther Clinically-Reportable CNVs\t17\t36460090\t37857812\t17q12\t17q12 duplication\tRecurrent CNV\thttps://www.ncbi.nlm.nih.gov/books/NBK344340/', 'recurrentCNV_hg38.oneway.tsv': 'Group (in Figure 2A)\tCHR\tSTART\tEND\tLocus name\tDisorder Name\tLocus type\tURL for Clinical Reference\nGenome-wide significant loci\t1\t145627235\t145809100\t1q21.1_TAR_BP1_BP2\t\tRecurrent CNV\t\nGenome-wide significant loci\t1\t145823900\t146040039\t1q21.1_TAR_BP2_BP3\t1q21.1 recurrent (TAR syndrome) region (proximal)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37428\nGenome-wide significant loci\t15\t22778537\t23067754\t15q11.2_BP1_BP2\t15q11.2 recurrent region (BP1-BP2) (includes NIPA1)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37448\nGenome-wide significant loci\t15\t28917240\t30083764\t15q13.1-13.2_BP3_BP4\t15q13 recurrent region (BP3-BP4) (includes APBA2)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-46285\nGenome-wide significant loci\t15\t31724867\t32170575\t15q13.3_CHRNA7\t\tRecurrent CNV\t\nGenome-wide significant loci\t16\t15417798\t16199832\t16p13.11_BP1_BP2\t16p13.11_BP1-BP2\tRecurrent CNV\thttps://www.rarechromo.org/media/information/Chromosome%2016/16p13.11%20microdeletions%20FTNW.pdf\nGenome-wide significant loci\t16\t16900000\t18050000\t16p13.11_BP2_BP3\t16p13.11_BP2-BP3\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37415\nGenome-wide significant loci\t16\t21938814\t22420568\t16p12.2_BP2_BP3\t16p12.2_BP2-BP3\tRecurrent CNV\thttps://www.ncbi.nlm.nih.gov/books/NBK274565/\nGenome-wide significant loci\t16\t28471892\t28590813\t16p11.2_BP1_BP2\t16p11.2_BP1-BP3\tRecurrent CNV\thttps://www.ncbi.nlm.nih.gov/books/NBK11167/\nGenome-wide significant loci\t16\t28811875\t29035462\t16p11.2_BP2_BP3\t16p11.2_BP2-BP3\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37486\nGenome-wide significant loci\t16\t29639519\t30189452\t16p11.2_BP4_BP5\t16p11.2 recurrent region (proximal BP4-BP5) (includes TBX6)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37400\nGenome-wide significant loci\t22\t19049819\t20250000\t22q11.21_A_B\t\tRecurrent CNV\t\nGenome-wide significant loci\t22\t20400000\t20665000\t22q11.21_B_C\t22q11.21_B-C\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37516\nGenome-wide significant loci\t22\t20750000\t21112437\t22q11.21_C_D\t22q11.21_C-D\tRecurrent CNV\t\nOther Clinically-Reportable CNVs\t22\t21565838\t22610000\t22q11.21_D_E\t22p11.21 (D-E)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37397\nOther Clinically-Reportable CNVs\t22\t22825000\t23311459\t22q11.21_E_F\t22p11.21 (E-F)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-37488\nOther Clinically-Reportable CNVs\t22\t23550000\t23950000\t22q11.21_F_G\t22p11.21 (F-G)\tRecurrent CNV\thttps://search.clinicalgenome.org/kb/gene-dosage/region/ISCA-46294\nOther Clinically-Reportable CNVs\t22\t24050000\t24200000\t22q11.21_G_H\t22p11.21 (G-H)\tRecurrent CNV\t\n\t15\t30788442\t31724867\t15q13.3_BP4_BP5_exclCHRNA7\t\t\t\n\t16\t21570000\t21720000\t16p12.2_BP1_BP2\t\t\t\n\t15\t23067754\t28145193\t15q11.2-13.1_BP2_BP3\t\t\t\n\t16\t29035462\t29639519\t16p11.2_BP3_BP4\t\t\t'}.items():
    open(f'{MG}/{f}', 'w').write(txt)
print(sorted(os.listdir(MG)))

In [ ]:
r10 = lambda n: f'~{int(round(n, -1))}' if n > MIN_CELL else mask(int(n))
norm = lambda x: re.sub(r'[^a-z0-9]', '', str(x).lower().replace('x_', '', 1))
import re
res = {}
for ds, cnv_, ph_ in [('wgs', f"{DATASETS['wgs']['run']}/data/aou_srwgs_sv.cnv", f"{DATASETS['wgs']['run']}/data/aou_pheno_std.tsv"),
                      ('array', f"{DATASETS['array']['run']}/data/aou_array.cnv", f"{DATASETS['array']['run']}/data/aou_array_pheno_std.tsv")]:
    out_ = f"{DATASETS[ds]['run']}/data/recurrent_genomatrix_molly.tsv"
    sh(f'{sys.executable} {MG}/recurrent_cnv_genotyping_aou.py --cnv {cnv_} --samples {ph_} --reciprocal {MG}/recurrentCNV_hg38.reciprocal.tsv '
       f'--oneway {MG}/recurrentCNV_hg38.oneway.tsv --out {out_} --min_cell {MIN_CELL}')
    G = pd.read_csv(out_, sep='\t', index_col=0)
    G = G[[c for c in G.columns if not c.endswith('_oneway')]]
    al = pd.read_csv(f"{DATASETS[ds]['run']}/results/all/taskA_alleles/height/allele_assoc.tsv", sep='\t')
    al = al.assign(key=al.allele.str.replace(r'_(DEL|DUP)$', '', regex=True).map(norm), typ=al.allele.str[-3:])
    rows = []
    for c in G.columns:
        r = dict(locus=c)
        for typ, code_ in [('DEL', 1), ('DUP', 3)]:
            r[f'{typ}_molly'] = r10(int((G[c] == code_).sum()))
            m = al[(al.key == norm(c)) & (al.typ == typ)]
            r[f'{typ}_alleles_py'] = (r10(int(pd.to_numeric(m.n_carriers, errors='coerce').fillna(0).iloc[0])) if str(m.n_carriers.iloc[0]).isdigit() else m.n_carriers.iloc[0]) if len(m) else '-'
        rows.append(r)
    res[ds] = pd.DataFrame(rows)
    print(f'\n=== {ds}: N = {len(G)} (people analysed); counts rounded to 10, 1-{MIN_CELL} masked; alleles_py = pipeline (all/height)')
    print(res[ds].to_string(index=False))